Langkah 1 - Persiapan Library

In [10]:
# !pip install pandas numpy openpyxl

import pandas as pd
import numpy as np

In [ ]:
#!pip install openpyxl

   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   ------ -------------------------------- 41.0/250.9 kB 653.6 kB/s eta 0:00:01
   -------------- ------------------------- 92.2/250.9 kB 1.1 MB/s eta 0:00:01
   ------------------------------------- -- 235.5/250.9 kB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 250.9/250.9 kB 1.7 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Langkah 2 - Membaca File Excel Mentah

In [11]:
# Tentukan jalur file menggunakan RELATIVE PATH
file_path = '../data/raw/Kompilasi Renaksi RB 2026 Hasil Penajaman Sekretariat.xlsx'

# Membaca file Excel pada sheet 'RB General'
df_general = pd.read_excel(file_path, sheet_name='RB General')

# Menampilkan 15 baris pertama untuk inspeksi awal
df_general.head(15)

,Rencana Aksi RB General Tahun 2026,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,No,Indikator Kegiatan Utama,Hasil Tahun 2025,Target Tahun 2026,Catatan dan Rekomendasi LHE Sementara RB 2025,Rencana Aksi,Output,NaN,Target Penyelesaian (Kumulatif),NaN,NaN,NaN,NaN,Anggaran,Ket,Unit/Satuan Kerja Pelaksana,NaN,Reviu Catatan,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Exante
3,NaN,NaN,NaN,NaN,NaN,NaN,Satuan,Indikator,TW 1,TW 2,TW 3,TW 4,Total,NaN,NaN,Koordinator (PJK),Pelaksana,Aksi → tujuan/outcome → dampak strategis,NaN,NaN
4,(1),(2),(3),(4),(5),(6),(7),(8),(9),(10),(11),(12),(13),(14),(15),(16),(17),NaN,NaN,NaN
5,1,Nilai SAKIP,"78,89","79,03",Badan Pusat Statistik perlu memperkuat keterka...,1. Diskusi dengan MenPANRB untuk penjelasan le...,Kegiatan,Laporan Rencana Tindak Lanjut rekomendasi AKIP...,0,1,1,1,1,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama,Rekomendasi Rencana Aksi melalui pola : \n\n1....,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,2. Benchmarking ke K/L untuk mendapatkan saran...,Kegiatan,Laporan benchmarking ke K/L,0,1,1,1,1,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,3. Penyusunan reviu Indikator Kinerja Utama 20...,Persen,Draf Reviu Perka IKU 2025-2029,25,50,75,100,100,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,4. Perbaikan CSF sehingga keterkaitan logis me...,Dokumen,Dokumen Perencanaan Kinerja yang memuat perbai...,0,0,1,1,1,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,5. Perbaikan dokumen pohon kinerja yang konsis...,Dokumen,Dokumen Perencanaan Kinerja yang memuat perbai...,0,0,1,1,1,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama,NaN,NaN,NaN


Langkah 3 - Melakukan ETL (Transformasi)

Tugasnya : 
1. Memotong : membuang baris 0-4 yang tidak berguna 
2. Mengganti nama : memberi nama kolom yang rapi sesuai standar database (huruf, kecil, menggunakan  underscore)
3. Menambal Lubang (forward fill) : Menggunakan ffill() untuk menarik teks dari atas ke bawah agar sel gabungan (NaN) terisi otomatis

In [12]:
# 1. Slicing: Kita ambil dari baris ke-5 sampai bawah, dan ambil 17 kolom pertama saja
df_clean = df_general.iloc[5:, :17].copy()

# 2. Renaming: Mengganti nama kolom agar rapi dan sesuai ERD kita
df_clean.columns = [
    'no', 'indikator_utama', 'hasil_2025', 'target_2026', 
    'catatan_lhe', 'rencana_aksi_desc', 'satuan', 'indikator_output', 
    'tw1', 'tw2', 'tw3', 'tw4', 'target_total', 
    'anggaran', 'keterangan', 'pic_koordinator', 'pic_pelaksana'
]

# 3. Forward Fill (ffill): Menambal nilai NaN di kolom induk akibat merged cells di Excel
kolom_induk = ['no', 'indikator_utama', 'hasil_2025', 'target_2026', 'catatan_lhe']
df_clean[kolom_induk] = df_clean[kolom_induk].ffill()

# 4. Cleansing tambahan: Membuang baris yang kolom rencana aksi-nya benar-benar kosong
df_clean = df_clean.dropna(subset=['rencana_aksi_desc'])

# Tampilkan hasilnya!
df_clean.head(10)

C:\Users\ABDILLAH FIKRI POME\AppData\Local\Temp\ipykernel_18508\1368269307.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean[kolom_induk] = df_clean[kolom_induk].ffill()


,no,indikator_utama,hasil_2025,target_2026,catatan_lhe,rencana_aksi_desc,satuan,indikator_output,tw1,tw2,tw3,tw4,target_total,anggaran,keterangan,pic_koordinator,pic_pelaksana
5,1,Nilai SAKIP,"78,89","79,03",Badan Pusat Statistik perlu memperkuat keterka...,1. Diskusi dengan MenPANRB untuk penjelasan le...,Kegiatan,Laporan Rencana Tindak Lanjut rekomendasi AKIP...,0,1,1,1,1,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama
6,1,Nilai SAKIP,"78,89","79,03",Badan Pusat Statistik perlu memperkuat keterka...,2. Benchmarking ke K/L untuk mendapatkan saran...,Kegiatan,Laporan benchmarking ke K/L,0,1,1,1,1,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama
7,1,Nilai SAKIP,"78,89","79,03",Badan Pusat Statistik perlu memperkuat keterka...,3. Penyusunan reviu Indikator Kinerja Utama 20...,Persen,Draf Reviu Perka IKU 2025-2029,25,50,75,100,100,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama
8,1,Nilai SAKIP,"78,89","79,03",Badan Pusat Statistik perlu memperkuat keterka...,4. Perbaikan CSF sehingga keterkaitan logis me...,Dokumen,Dokumen Perencanaan Kinerja yang memuat perbai...,0,0,1,1,1,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama
9,1,Nilai SAKIP,"78,89","79,03",Badan Pusat Statistik perlu memperkuat keterka...,5. Perbaikan dokumen pohon kinerja yang konsis...,Dokumen,Dokumen Perencanaan Kinerja yang memuat perbai...,0,0,1,1,1,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama
10,1,Nilai SAKIP,"78,89","79,03",Perlu dilakukan penguatan dan keberlanjutan pr...,1. Penyusunan croscutting kinerja yang memuat ...,Dokumen,Dokumen Perencanaan Kinerja yang memuat crossc...,0,0,1,1,1,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama
11,1,Nilai SAKIP,"78,89","79,03",Setiap unit kerja perlu melakukan reviu dan ti...,1. Pemeriksaan Kertas Kerja Monitoring Kinerja...,Dokumen,Dokumen Kertas Kerja Pemeriksaan Monev SAKIP T...,1,2,3,4,4,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama
12,1,Nilai SAKIP,"78,89","79,03",Setiap unit kerja perlu melakukan reviu dan ti...,2. Penyesuaian proses bisnis Pembahasan Rapat ...,Dokumen,Template notula rapat monitoring kinerja triwu...,0,0,1,2,2,NaN,NaN,Biro Perencanaan dan Kerjasama,Biro Perencanaan dan Kerjasama
13,1,Nilai SAKIP,"78,89","79,03",Mendorong BPS untuk melakukan percepatan integ...,1. Penggunaan fitur Perencanaan Kinerja dan si...,Persen,Persentase data PK yg disinkronkan dari SINERG...,100,100,100,100,100,NaN,NaN,Biro Perencanaan dan Kerjasama,1.Biro Perencanaan dan Kerjasama
14,1,Nilai SAKIP,"78,89","79,03",Instansi perlu memastikan konsistensi dan peme...,1. Penyusunan pedoman teknis Pelaporan Kinerja...,Dokumen,Dokumen pedoman teknis Laporan Kinerja BPS,1,1,1,2,2,NaN,NaN,Biro Perencanaan dan Kerjasama,1.Biro Perencanaan dan Kerjasama\n2. Dir SIS


Langkah 4 - MELT (Unpivot): Mengubah kolom TW berjejer menjadi baris memanjang ke bawah

In [13]:
#ini sebagai cikal bakal tabel 'target_output' di ERD
df_target = pd.melt(
    df_clean, 
    id_vars=['no', 'rencana_aksi_desc', 'satuan', 'indikator_output'], # Kolom jangkar (tidak ikut dilelehkan)
    value_vars=['tw1', 'tw2', 'tw3', 'tw4'],                           # Kolom yang akan dilelehkan
    var_name='periode_tw',                                             # Nama kolom baru untuk nama TW
    value_name='target_penyelesaian'                                   # Nama kolom baru untuk nilai angkanya
)

# Urutkan berdasarkan nomor kegiatan dan periode agar enak dibaca
df_target = df_target.sort_values(by=['no', 'periode_tw']).reset_index(drop=True)

# Tampilkan hasilnya!
df_target.head(12)

,no,rencana_aksi_desc,satuan,indikator_output,periode_tw,target_penyelesaian
0,1,1. Diskusi dengan MenPANRB untuk penjelasan le...,Kegiatan,Laporan Rencana Tindak Lanjut rekomendasi AKIP...,tw1,0
1,1,2. Benchmarking ke K/L untuk mendapatkan saran...,Kegiatan,Laporan benchmarking ke K/L,tw1,0
2,1,3. Penyusunan reviu Indikator Kinerja Utama 20...,Persen,Draf Reviu Perka IKU 2025-2029,tw1,25
3,1,4. Perbaikan CSF sehingga keterkaitan logis me...,Dokumen,Dokumen Perencanaan Kinerja yang memuat perbai...,tw1,0
4,1,5. Perbaikan dokumen pohon kinerja yang konsis...,Dokumen,Dokumen Perencanaan Kinerja yang memuat perbai...,tw1,0
5,1,1. Penyusunan croscutting kinerja yang memuat ...,Dokumen,Dokumen Perencanaan Kinerja yang memuat crossc...,tw1,0
6,1,1. Pemeriksaan Kertas Kerja Monitoring Kinerja...,Dokumen,Dokumen Kertas Kerja Pemeriksaan Monev SAKIP T...,tw1,1
7,1,2. Penyesuaian proses bisnis Pembahasan Rapat ...,Dokumen,Template notula rapat monitoring kinerja triwu...,tw1,0
8,1,1. Penggunaan fitur Perencanaan Kinerja dan si...,Persen,Persentase data PK yg disinkronkan dari SINERG...,tw1,100
9,1,1. Penyusunan pedoman teknis Pelaporan Kinerja...,Dokumen,Dokumen pedoman teknis Laporan Kinerja BPS,tw1,1


Langkah 5 - MEMECAH TABEL SESUAI ERD & EXPORT KE CSV

In [14]:
# A. Membuat Tabel Master 'Rencana Aksi' 
# hanya mengaambil kolom utamanya saja, tanpa data target/TW karena sudah ada di df_target
kolom_master = [
    'no', 'indikator_utama', 'hasil_2025', 'target_2026', 
    'catatan_lhe', 'rencana_aksi_desc', 'pic_koordinator', 'pic_pelaksana'
]
df_rencana_aksi = df_clean[kolom_master].drop_duplicates().reset_index(drop=True)

# B. Menyimpan (Export) kedua tabel menjadi file CSV ke folder processed
df_rencana_aksi.to_csv('../data/processed/master_rencana_aksi.csv', index=False)
df_target.to_csv('../data/processed/tabel_target_output.csv', index=False)

print("Proses ETL Selesai......")

Proses ETL Selesai......


MELAKUKAN UNTUK SEMUA RENCANA AKSI (GENERAL DAN TEMATIK(ADA 6 TEMATIK))

In [15]:
# 1. Daftar semua sheet yang ingin diproses
sheet_names = [
    'RB General', 'RB Tematik 1. Kemiskinan_', 
    'RB Tematik 2. Investasi_', 'RB Tematik 3. Hilirisasi_', 
    'RB Tematik 4. Kesehatan_', 'RB Tematik 5. Pangan_', 
    'RB Tematik 6. Pendidikan_'
]

all_data = []

# 2. Loop untuk memproses setiap sheet secara otomatis
for sheet in sheet_names:
    print(f"Memproses sheet: {sheet}...")
    
    # Membaca sheet
    temp_df = pd.read_excel(file_path, sheet_name=sheet)
    
    # Slicing (asumsi struktur baris awal tetap sama)
    temp_df = temp_df.iloc[5:, :17].copy()
    
    # Beri nama kolom (menggunakan nama standar yang sudah kita sepakati)
    temp_df.columns = [
        'no', 'indikator_utama', 'hasil_2025', 'target_2026', 
        'catatan_lhe', 'rencana_aksi_desc', 'satuan', 'indikator_output', 
        'tw1', 'tw2', 'tw3', 'tw4', 'target_total', 
        'anggaran', 'keterangan', 'pic_koordinator', 'pic_pelaksana'
    ]
    
    # Tambahkan kolom penanda asal sheet agar tahu data ini dari tematik mana
    temp_df['kategori_tematik'] = sheet
    
    # Forward Fill untuk menambal NaN
    kolom_isi = ['no', 'indikator_utama', 'hasil_2025', 'target_2026', 'catatan_lhe']
    temp_df[kolom_isi] = temp_df[kolom_isi].ffill()
    
    # Bersihkan baris kosong
    temp_df = temp_df.dropna(subset=['rencana_aksi_desc'])
    
    # Masukkan ke list all_data
    all_data.append(temp_df)

# 3. Gabungkan semua menjadi satu tabel besar (Master Data)
df_final = pd.concat(all_data, ignore_index=True)

# 4. Simpan hasil akhir yang sudah terintegrasi
df_final.to_csv('../data/processed/master_renaksi_bps_terintegrasi.csv', index=False)
print("🎉 Semua sheet berhasil diproses dan digabungkan menjadi 'master_renaksi_bps_terintegrasi.csv'!")

Memproses sheet: RB General...
Memproses sheet: RB Tematik 1. Kemiskinan_...


C:\Users\ABDILLAH FIKRI POME\AppData\Local\Temp\ipykernel_18508\1135216488.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df[kolom_isi] = temp_df[kolom_isi].ffill()
C:\Users\ABDILLAH FIKRI POME\AppData\Local\Temp\ipykernel_18508\1135216488.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df[kolom_isi] = temp_df[kolom_isi].ffill()


Memproses sheet: RB Tematik 2. Investasi_...
Memproses sheet: RB Tematik 3. Hilirisasi_...
Memproses sheet: RB Tematik 4. Kesehatan_...


C:\Users\ABDILLAH FIKRI POME\AppData\Local\Temp\ipykernel_18508\1135216488.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df[kolom_isi] = temp_df[kolom_isi].ffill()
C:\Users\ABDILLAH FIKRI POME\AppData\Local\Temp\ipykernel_18508\1135216488.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df[kolom_isi] = temp_df[kolom_isi].ffill()
C:\Users\ABDILLAH FIKRI POME\AppData\Local\Temp\ipykernel_18508\1135216488.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. C

Memproses sheet: RB Tematik 5. Pangan_...
Memproses sheet: RB Tematik 6. Pendidikan_...
🎉 Semua sheet berhasil diproses dan digabungkan menjadi 'master_renaksi_bps_terintegrasi.csv'!


C:\Users\ABDILLAH FIKRI POME\AppData\Local\Temp\ipykernel_18508\1135216488.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df[kolom_isi] = temp_df[kolom_isi].ffill()
